# Anomaly Detection for Energy Analytics - NSP Dataset

## Objective
Implement multiple anomaly detection methods and export results for Tableau dashboard.

Required output: `anomaly_results.csv` with columns:
- timestamp
- region
- consumption_kwh
- anomaly_flag
- anomaly_type
- anomaly_method

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Helper function
def save_and_show_plot(fig, filename):
    output_path = OUTPUT_DIR / filename
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()

print("Setup complete!")

In [ ]:
# Load engineered features
df = pd.read_csv(OUTPUT_DIR / 'engineered_features.csv', parse_dates=['timestamp'])

# Reconstruct region column if needed
if 'region' not in df.columns:
    region_cols = [col for col in df.columns if col.startswith('region_')]
    df['region'] = df[region_cols].idxmax(axis=1).str.replace('region_', '')

print(f"Dataset shape: {df.shape}")
print(f"Regions: {sorted(df['region'].unique())}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Check existing anomaly_flag column
if 'anomaly_flag' in df.columns:
    print(f"\nExisting anomaly_flag distribution:")
    print(df['anomaly_flag'].value_counts())
    print(f"Anomaly rate: {df['anomaly_flag'].sum() / len(df) * 100:.2f}%")

df.head()

## Understanding Existing Anomalies

Let's analyze the pre-labeled anomalies in the dataset before implementing our own detection methods.

In [ ]:
# Analyze existing anomalies by region
anomaly_by_region = df.groupby('region')['anomaly_flag'].agg(['sum', 'count', 'mean'])
anomaly_by_region.columns = ['Anomaly_Count', 'Total_Records', 'Anomaly_Rate']
anomaly_by_region['Anomaly_Rate'] = anomaly_by_region['Anomaly_Rate'] * 100

print("Existing Anomalies by Region:")
print(anomaly_by_region)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of counts
anomaly_by_region['Anomaly_Count'].plot(kind='bar', ax=axes[0], color='coral')
axes[0].set_title('Anomaly Count by Region')
axes[0].set_ylabel('Number of Anomalies')
axes[0].set_xlabel('Region')
axes[0].tick_params(axis='x', rotation=45)

# Anomaly rate
anomaly_by_region['Anomaly_Rate'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Anomaly Rate by Region (%)')
axes[1].set_ylabel('Anomaly Rate (%)')
axes[1].set_xlabel('Region')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
save_and_show_plot(fig, 'existing_anomalies_by_region.png')

## Statistical Method 1: Z-Score Detection

Detects anomalies based on standard deviations from the mean.
- Threshold: |z-score| > 3
- Applied per region to account for regional differences

In [ ]:
def detect_zscore_anomalies(df, column='consumption_kwh', threshold=3):
    """Detect anomalies using Z-score method per region"""
    df_copy = df.copy()
    df_copy['zscore_anomaly'] = 0
    
    for region in df_copy['region'].unique():
        mask = df_copy['region'] == region
        values = df_copy.loc[mask, column]
        
        # Calculate z-scores
        z_scores = np.abs(stats.zscore(values, nan_policy='omit'))
        
        # Mark anomalies
        df_copy.loc[mask, 'zscore_anomaly'] = (z_scores > threshold).astype(int)
    
    return df_copy

# Apply Z-score detection
df = detect_zscore_anomalies(df)

zscore_stats = df.groupby('region')['zscore_anomaly'].agg(['sum', 'mean'])
zscore_stats.columns = ['Anomaly_Count', 'Anomaly_Rate']
zscore_stats['Anomaly_Rate'] = zscore_stats['Anomaly_Rate'] * 100

print("Z-Score Anomaly Detection Results:")
print(zscore_stats)
print(f"\nTotal anomalies detected: {df['zscore_anomaly'].sum()} ({df['zscore_anomaly'].mean()*100:.2f}%)")

## Statistical Method 2: IQR (Interquartile Range) Detection

Robust to outliers, uses quartile-based thresholds.
- Anomalies: values < Q1 - 1.5×IQR or > Q3 + 1.5×IQR
- Applied per region

In [ ]:
def detect_iqr_anomalies(df, column='consumption_kwh', multiplier=1.5):
    """Detect anomalies using IQR method per region"""
    df_copy = df.copy()
    df_copy['iqr_anomaly'] = 0
    
    for region in df_copy['region'].unique():
        mask = df_copy['region'] == region
        values = df_copy.loc[mask, column]
        
        # Calculate IQR
        Q1 = values.quantile(0.25)
        Q3 = values.quantile(0.75)
        IQR = Q3 - Q1
        
        # Define bounds
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        # Mark anomalies
        anomalies = (values < lower_bound) | (values > upper_bound)
        df_copy.loc[mask, 'iqr_anomaly'] = anomalies.astype(int)
    
    return df_copy

# Apply IQR detection
df = detect_iqr_anomalies(df)

iqr_stats = df.groupby('region')['iqr_anomaly'].agg(['sum', 'mean'])
iqr_stats.columns = ['Anomaly_Count', 'Anomaly_Rate']
iqr_stats['Anomaly_Rate'] = iqr_stats['Anomaly_Rate'] * 100

print("IQR Anomaly Detection Results:")
print(iqr_stats)
print(f"\nTotal anomalies detected: {df['iqr_anomaly'].sum()} ({df['iqr_anomaly'].mean()*100:.2f}%)")